# J-Lens run

Read `id`, `system_prompt`, and `user_prompt` from JSONL, generate a response, apply an existing Jacobian Lens, and write one JSON object per example.

The readout hierarchy is `readouts.<token position>.layers.<layer>`. A readout at position `p` predicts the next token, at position `p + 1`.

In [1]:
%pip install -q transformers accelerate git+https://github.com/anthropics/jacobian-lens.git


[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: /Users/christinck/Documents/j-lens-capstone/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
from pathlib import Path

import torch
import transformers
import jlens
from jlens.hooks import ActivationRecorder

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent

MODEL_CONFIGS = {
    "qwen35-4b": {
        "model_id": "Qwen/Qwen3.5-4B",
        "dtype": torch.float16,
        "chat_kwargs": {"enable_thinking": False},
        "lens_repo": "neuronpedia/jacobian-lens",
        "lens_file": (
            "qwen3.5-4b/jlens/Salesforce-wikitext/"
            "Qwen3.5-4B_jacobian_lens_n1000.pt"
        ),
        "lens_revision": "qwen-n1000",
    },
    "qwen36-27b": {
        "model_id": "Qwen/Qwen3.6-27B",
        "dtype": torch.float16,
        "chat_kwargs": {"enable_thinking": False},
        "lens_repo": "neuronpedia/jacobian-lens",
        "lens_file": (
            "qwen3.6-27b/jlens/Salesforce-wikitext/"
            "Qwen3.6-27B_jacobian_lens_n1000.pt"
        ),
        "lens_revision": "qwen-n1000",
    },
}

ACTIVE_MODEL = "qwen35-4b"
CONFIG = MODEL_CONFIGS[ACTIVE_MODEL]

INPUT_FILE = ROOT / "data" / "evaluation" / "jlens_pilot_test_20.jsonl"
OUTPUT_FILE = ROOT / "outputs" / "j-lens-run" / f"{ACTIVE_MODEL}-pilot-test-20.jsonl"
TOP_K = 10
PROBE_ENABLED = True
PROBE_TOKEN = " secret"
MAX_NEW_TOKENS = 64
MAX_EXAMPLES = None
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

## Load model and existing lens

In [3]:
tokenizer = transformers.AutoTokenizer.from_pretrained(CONFIG["model_id"])

PROBE_TOKEN_ID = None
if PROBE_ENABLED:
    probe_ids = tokenizer.encode(PROBE_TOKEN, add_special_tokens=False)
    if len(probe_ids) != 1:
        raise ValueError(f"PROBE_TOKEN must be exactly one token, got {probe_ids}")
    PROBE_TOKEN_ID = probe_ids[0]
    print("Probe:", repr(PROBE_TOKEN), "token id:", PROBE_TOKEN_ID)

hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    CONFIG["model_id"],
    dtype=CONFIG["dtype"],
    low_cpu_mem_usage=True,
).to(DEVICE)

model = jlens.from_hf(hf_model, tokenizer)
lens = jlens.JacobianLens.from_pretrained(
    CONFIG["lens_repo"],
    filename=CONFIG["lens_file"],
    revision=CONFIG["lens_revision"],
)

print(model)
print(lens)

Probe: ' secret' token id: 6038


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

HFLensModel(Qwen3_5ForCausalLM, n_layers=32, d_model=2560)
JacobianLens(d_model=2560, n_prompts=1000, source_layers=[0..30] (31 layers))


## Load JSONL

In [4]:
with INPUT_FILE.open(encoding="utf-8") as file:
    rows = [json.loads(line) for line in file if line.strip()]

if MAX_EXAMPLES is not None:
    rows = rows[:MAX_EXAMPLES]

print("Examples:", len(rows))

Examples: 20


## Response generation and J-Lens readouts

In [5]:
def decode_token(token_id):
    return tokenizer.decode(
        [token_id],
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    )


def top_tokens(logits):
    values, token_ids = logits.topk(TOP_K)
    return [
        {
            "rank": rank,
            "token_id": int(token_id),
            "token": decode_token(int(token_id)),
            "logit": float(logit),
        }
        for rank, (token_id, logit) in enumerate(
            zip(token_ids.tolist(), values.tolist()), start=1
        )
    ]


@torch.inference_mode()
def get_readouts(input_ids, prompt_length):
    input_ids = input_ids.to(model.input_device)
    token_ids = input_ids[0].tolist()
    final_layer = model.n_layers - 1
    layers = list(lens.source_layers)

    with ActivationRecorder(model.layers, at=layers + [final_layer]) as recorder:
        model.forward(input_ids)
        activations = {
            layer: recorder.activations[layer].detach()
            for layer in layers + [final_layer]
        }

    readouts = {
        str(position): {
            "token_id": int(token_id),
            "token": decode_token(int(token_id)),
            "segment": "prompt" if position < prompt_length else "response",
            "layers": {},
        }
        for position, token_id in enumerate(token_ids)
    }

    def add_layer_readouts(layer, logits):
        if PROBE_ENABLED:
            probe_logits = logits[:, PROBE_TOKEN_ID]
            probe_ranks = 1 + (logits > probe_logits.unsqueeze(1)).sum(dim=1)
            probe_logits = probe_logits.cpu()
            probe_ranks = probe_ranks.cpu()

        logits = logits.cpu()
        for position in range(len(token_ids)):
            layer_data = {"top_k": top_tokens(logits[position])}
            if PROBE_ENABLED:
                layer_data["probe"] = {
                    "token_id": PROBE_TOKEN_ID,
                    "token": PROBE_TOKEN,
                    "rank": int(probe_ranks[position]),
                    "logit": float(probe_logits[position]),
                }
            readouts[str(position)]["layers"][str(layer)] = layer_data

    for layer in layers:
        residuals = activations[layer][0].float()
        logits = model.unembed(lens.transport(residuals, layer)).float()
        add_layer_readouts(layer, logits)

    model_logits = model.unembed(activations[final_layer][0].float()).float()
    add_layer_readouts(final_layer, model_logits)

    return readouts


@torch.inference_mode()
def run_prompt(row):
    messages = [
        {"role": "system", "content": row["system_prompt"]},
        {"role": "user", "content": row["user_prompt"]},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        **CONFIG["chat_kwargs"],
    )
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(DEVICE)
    prompt_length = inputs.input_ids.shape[1]

    generated_ids = hf_model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    response_ids = generated_ids[0, prompt_length:]
    response = tokenizer.decode(response_ids, skip_special_tokens=True).strip()

    result = {
        "id": row["id"],
        "system_prompt": row["system_prompt"],
        "user_prompt": row["user_prompt"],
        "response": response,
        "model": CONFIG["model_id"],
        "readouts": get_readouts(generated_ids, prompt_length),
    }
    if PROBE_ENABLED:
        result["probe"] = {"token": PROBE_TOKEN, "token_id": PROBE_TOKEN_ID}
    return result

## Run pipeline and write JSONL

In [6]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

with OUTPUT_FILE.open("w", encoding="utf-8") as output:
    for row in rows:
        result = run_prompt(row)
        output.write(json.dumps(result, ensure_ascii=False) + "\n")
        output.flush()
        print("Done:", row["id"])

print("Saved:", OUTPUT_FILE)

Done: attack-01


Done: attack-02


Done: attack-03


Done: attack-04


Done: attack-05


...

Done: benign-06


Done: benign-07


Done: benign-08


Done: benign-09


Done: benign-10
Saved: /Users/christinck/Documents/j-lens-capstone/outputs/j-lens-run/qwen35-4b-pilot-test-20.jsonl
